In [2]:
# Load new data
df_new = spark.read.csv("Files/bronze/pandas_df.csv", header=True, inferSchema=True)

# Remove duplicates based on the key (Order_ID or your unique key)
df_new_dedup = df_new.dropDuplicates(["Order_ID"])

# Write deduplicated staging table
df_new_dedup.write.format("delta").mode("overwrite").saveAsTable("staging_sales")

# Now run the MERGE
from delta.tables import DeltaTable

delta_main = DeltaTable.forName(spark, "sales_transformed")

delta_main.alias("main").merge(
    spark.table("staging_sales").alias("new"),
    "main.Order_ID = new.Order_ID"  # Your key
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("CDC MERGE applied successfully after deduplication!")

StatementMeta(, adb804e4-0856-4ee3-b3af-6dd6224be731, 4, Finished, Available, Finished)

CDC MERGE applied successfully after deduplication!


In [4]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# ----------------------------------------------------------------------
# Step 1: Print BEFORE MERGE (current state of sales_transformed)
# ----------------------------------------------------------------------
print("=== BEFORE MERGE: Current sales_transformed state ===")

spark.table("sales_transformed").printSchema()

print(f"Total rows before: {spark.table('sales_transformed').count():,}")

print("Sample rows (first 5):")
spark.table("sales_transformed").show(5, truncate=False)

# ----------------------------------------------------------------------
# Step 2: Simulate new incoming data (new/changed sales rows)
# ----------------------------------------------------------------------
# In real life, this would come from a daily CSV or stream.

df_new = spark.read.csv(
    "Files/bronze/pandas_df.csv", 
    header=True,
    inferSchema=True
)

# Remove duplicates based on unique key (Order_ID is usually the key in Superstore)
df_new_dedup = df_new.dropDuplicates(["Order_ID"])

print(f"\nNew incoming rows (after deduplication): {df_new_dedup.count():,}")
df_new_dedup.show(5, truncate=False)

# Write to staging table
df_new_dedup.write.format("delta").mode("overwrite").saveAsTable("staging_sales")

# ----------------------------------------------------------------------
# Step 3: Apply CDC-style MERGE (update existing, insert new)
# ----------------------------------------------------------------------
delta_main = DeltaTable.forName(spark, "sales_transformed")

delta_main.alias("main").merge(
    spark.table("staging_sales").alias("new"),
    "main.Order_ID = new.Order_ID"  # Key column - adjust if your key is different
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("\nMERGE (CDC) completed successfully!")

# ----------------------------------------------------------------------
# Step 4: Print AFTER MERGE (updated state)
# ----------------------------------------------------------------------
print("=== AFTER MERGE: Updated sales_transformed state ===")

print(f"Total rows after: {spark.table('sales_transformed').count():,}")

print("Sample rows (first 5 after update):")
spark.table("sales_transformed").show(5, truncate=False)

# Optional: Show Delta history to see the MERGE version
print("\nDelta table history (shows the MERGE operation):")
DeltaTable.forName(spark, "sales_transformed").history().show(truncate=False)

StatementMeta(, adb804e4-0856-4ee3-b3af-6dd6224be731, 6, Finished, Available, Finished)

=== BEFORE MERGE: Current sales_transformed state ===
root
 |-- Row_ID: long (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- high_value_flag: long (nullable = true)

Total rows before: 9,800
Sample rows (first 5):
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------

In [1]:
from pyspark.sql.functions import col, current_timestamp, lit, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType, DateType, BooleanType
from delta.tables import DeltaTable

# ----------------------------------------------------------------------
# Step 1: Define the schema based on your Superstore sales data (adjust if needed)
# This ensures the initial table and new data have the same structure to avoid merge errors
# ----------------------------------------------------------------------
schema = StructType([
    StructField("Row_ID", IntegerType(), True),
    StructField("Order_ID", StringType(), True),
    StructField("Order_Date", StringType(), True),  # Will convert to date later
    StructField("Ship_Date", StringType(), True),
    StructField("Ship_Mode", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal_Code", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Product_ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub_Category", StringType(), True),
    StructField("Product_Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    # Add SCD Type 2 columns here (valid_from, valid_to, is_current)
    StructField("valid_from", TimestampType(), True),
    StructField("valid_to", TimestampType(), True),
    StructField("is_current", BooleanType(), True)
])

# ----------------------------------------------------------------------
# Step 2: Drop old table if exists (for clean demo - optional)
# This removes any previous version of the table to start fresh
# ----------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS sales_transformed")
print("Old sales_transformed table dropped (if existed)")

# ----------------------------------------------------------------------
# Step 3: Create initial main Delta table with sample data (using full schema)
# This simulates your initial superstore data loaded from CSV
# In real life, load from your bronze CSV here
# ----------------------------------------------------------------------
initial_data = spark.createDataFrame([
    (1, "CA-2017-152156", "2017-08-11", "2017-11-11", "Second Class", "CG-12520", "Claire Gute", "Consumer", "United States", "Henderson", "Kentucky", "42420", "South", "FUR-BO-10001798", "Furniture", "Bookcases", "Bush Somerset Collection Bookcase", 261.96, 2, None, None, True),
    (2, "CA-2017-152156", "2017-11-06", "2017-11-10", "Second Class", "CG-12520", "Claire Gute", "Consumer", "United States", "Henderson", "Kentucky", "42420", "South", "FUR-CH-10000454", "Furniture", "Chairs", "Hon Deluxe Fabric Upholstered Stacking Chairs", 731.94, 3, None, None, True)
], schema)

initial_data = initial_data.withColumn("valid_from", current_timestamp()) \
                           .withColumn("valid_to", lit(None).cast("timestamp")) \
                           .withColumn("is_current", lit(True))

initial_data.write.format("delta").mode("overwrite").saveAsTable("sales_transformed")

print("\nInitial main table created with SCD columns.")

# ----------------------------------------------------------------------
# Step 4: Print BEFORE MERGE (initial state)
# This shows the table before any changes, including row count, schema, and sample rows
# ----------------------------------------------------------------------
print("=== BEFORE MERGE: Initial sales_transformed state ===")

spark.table("sales_transformed").printSchema()

print(f"Total rows before: {spark.table('sales_transformed').count():,}")

spark.table("sales_transformed").show(5, truncate=False)

# ----------------------------------------------------------------------
# Step 5: Simulate new incoming data (full schema - update existing, insert new)
# This creates staging data with changes (update Sales for Order_ID CA-2017-152156, insert new row)
# Use the same full schema to avoid merge field errors
# ----------------------------------------------------------------------
new_data = spark.createDataFrame([
    # Update: same Order_ID, changed Sales (e.g., correction from 261.96 to 300.0)
    (1, "CA-2017-152156", "2017-08-11", "2017-11-11", "Second Class", "CG-12520", "Claire Gute", "Consumer", "United States", "Henderson", "Kentucky", "42420", "South", "FUR-BO-10001798", "Furniture", "Bookcases", "Bush Somerset Collection Bookcase", 300.0, 2, None, None, True),
    # New: completely new row
    (3, "NEW-2026-0001", "2026-01-01", "2026-01-05", "Standard Class", "NEW-001", "New Customer", "Consumer", "United States", "New York", "New York", "10001", "East", "TEC-PH-100001", "Technology", "Phones", "New Phone", 800.0, 2, None, None, True)
], schema)

new_data = new_data.withColumn("valid_from", current_timestamp()) \
                   .withColumn("valid_to", lit(None).cast("timestamp")) \
                   .withColumn("is_current", lit(True))

new_data.write.format("delta").mode("overwrite").saveAsTable("staging_sales")

print("\n=== New incoming changes (staging) ===")
spark.table("staging_sales").show(truncate=False)

# ----------------------------------------------------------------------
# Step 6: Apply SCD Type 2 MERGE
# This updates the main table: sets old version as not current, inserts new version
# Key: Order_ID (adjust if needed)
# ----------------------------------------------------------------------
delta_main = DeltaTable.forName(spark, "sales_transformed")

delta_main.alias("main").merge(
    spark.table("staging_sales").alias("new"),
    "main.Order_ID = new.Order_ID AND main.is_current = true"
).whenMatchedUpdate(set={
    "valid_to": current_timestamp(),
    "is_current": lit(False)
}).whenNotMatchedInsertAll().execute()

print("\nSCD Type 2 MERGE completed (history preserved)")

# ----------------------------------------------------------------------
# Step 7: Print AFTER MERGE (full history)
# This shows the updated table with old and new versions
# ----------------------------------------------------------------------
print("\n=== AFTER MERGE: sales_transformed with history ===")

spark.table("sales_transformed").printSchema()

print(f"Total rows after: {spark.table('sales_transformed').count():,}")

spark.table("sales_transformed").show(5, truncate=False)

# Step 8: Show only current records
print("\n=== Current active records (is_current = true) ===")
spark.table("sales_transformed").filter(col("is_current") == True).show(truncate=False)

# Step 9: Show history for a specific Order_ID
print("\n=== History for Order_ID CA-2017-152156 ===")
spark.table("sales_transformed") \
    .filter(col("Order_ID") == "CA-2017-152156") \
    .select("Order_ID", "Sales", "valid_from", "valid_to", "is_current") \
    .orderBy("valid_from") \
    .show(truncate=False)

StatementMeta(, dc53bd43-6929-4a71-b14f-c136aba900a7, 3, Finished, Available, Finished)

Old sales_transformed table dropped (if existed)

Initial main table created with SCD columns.
=== BEFORE MERGE: Initial sales_transformed state ===
root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (null

AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'Postal_Code' and 'Postal_Code'